# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring** — ranking which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring.

I chose Lane 2 because it has the clearest customer and the strongest evidence already in hand. The question is a decision someone makes every week — "which page do I look at first?" — not an abstract description of the data. It is also the lane the starter pipeline was built for: baseline score, reason codes, a ranked queue, and precision@K. The numbers below show a large, real review pool and a learned ranking that beats the fixed rule on the starter slice.

In [6]:
import os
import pandas as pd

candidates = [
    "../../data/raw/content_refresh_anonymized.csv",  # run from work/notebooks/
    "data/raw/content_refresh_anonymized.csv",        # run from repo root
]
path = next(p for p in candidates if os.path.exists(p))
df = pd.read_csv(path)

required = ["impressions_90d", "sessions_90d", "trend_direction",
            "ctr", "avg_position", "content_age_days"]
missing = [c for c in required if c not in df.columns]
print("Lane 2 required columns:", required)
print("Missing:", missing if missing else "none")


Lane 2 required columns: ['impressions_90d', 'sessions_90d', 'trend_direction', 'ctr', 'avg_position', 'content_age_days']
Missing: none


## 2. The question: decision, action, cost of a wrong call

**Decision.** Which page should a content editor review next for refresh?

**Action.** The editor works a ranked queue top-down; each row carries a reason code (e.g. `declining_with_demand`, `stale_visible_page`) so they see why a page ranked there before opening it. The queue matches real capacity — review the top 20 or top 50, not all 30,000 rows.

**Cost of a wrong call.** Two ways to be wrong, and both cost real hours:
- *False positive (a dud ranked high):* the editor spends limited review time on a page that will not recover and a better candidate slides down the queue. This is the expensive error, so the metric is precision@K, not accuracy.
- *False negative (a declining page ranked low):* the page keeps losing impressions and sessions for weeks while nothing is done — a compounding loss.

**Why data or ML helps.** Because the pattern is real but too tangled to hand-write: on the starter slice, under client-holdout, the random forest reached precision@50 of 0.740 (37 of its top 50 right) versus the fixed rule's 0.240 (12 of 50). A plain if-statement cannot separate the pages that matter from the noise.

In [7]:
# The decision surface: pages an editor could actually act on.
visible = (df["impressions_90d"] >= 500).sum()
declining_visible = ((df["impressions_90d"] >= 500) & (df["trend_direction"] == "down")).sum()

print("Visible pages (>=500 impressions in 90d):", visible)
print("Visible AND declining:", declining_visible)
print("That is the review queue this lane ranks.")


Visible pages (>=500 impressions in 90d): 16726
Visible AND declining: 9961
That is the review queue this lane ranks.


## 3. Quick look at the data (2-3 real numbers)

The numbers below are computed live from the starter CSV in the code cell, not pasted from memory. Three of them make this lane look worth it:

1. **30,000 pages across 32 pseudonymized clients** — a large enough review pool that ranking matters.
2. **About 54% of pages are currently declining** — the problem is not rare.
3. **Thousands of visible pages (≥ 500 impressions in 90d) are also declining** — real pages with real demand an editor could act on: exactly the `declining_with_demand` surface Lane 2 ranks.

One more piece of evidence from the starter pipeline: a learned ranking beat the fixed rule on this slice — precision@50 0.240 → 0.740 under client-holdout. That says the signal is there, but it also reminds me the starter label is a current-window bucket — a *proxy*. A stronger capstone target is a future-window outcome measured on the warehouse data.

In [8]:
import os
import pandas as pd

candidates = [
    "../../data/raw/content_refresh_anonymized.csv",  # run from work/notebooks/
    "data/raw/content_refresh_anonymized.csv",        # run from repo root
]
path = next(p for p in candidates if os.path.exists(p))
df = pd.read_csv(path)

print("Starter slice:", df.shape[0], "rows x", df.shape[1], "columns;",
      df["client_id"].nunique(), "pseudonymized clients")

declining_rate = (df["trend_direction"] == "down").mean()
print("Share of pages currently declining:", f"{declining_rate:.1%}")

pool = ((df["impressions_90d"] >= 500) & (df["trend_direction"] == "down")).sum()
print("Visible pages that are also declining:", pool)

no_position = (df["avg_position"] == 0).sum()
print("Rows with avg_position == 0 (no position data, not rank zero):", no_position)


Starter slice: 30000 rows x 44 columns; 32 pseudonymized clients
Share of pages currently declining: 54.2%
Visible pages that are also declining: 9961
Rows with avg_position == 0 (no position data, not rank zero): 1205


## 4. Careful words: what I can and can't claim

**What this work can say (observed / decision-support):**
- The queue ranks candidates for a human reviewer — decision support, not an automatic publishing decision.
- "Pages like these tend to be declining" — an observed association on anonymized data, not proof that a refresh causes a recovery.
- Precision@K measured on a held-out set — a real, repeatable number.

**What it never can say:**
- *Causal proof.* No experiment, so I will never claim that editing a page caused a recovery.
- *Predicting Google.* These data describe what was observed; they do not reveal ranking rules.
- *Anything from the label's own inputs.* `trend_direction` and `trend_pct` derive the label and are never features (the guard in the code cell below).

The starter label itself is a current-window bucket, so I will call it a proxy — not the ideal capstone target. If I keep this lane, the capstone should use a future-window outcome (features from a prior window, outcome from a later window) from the warehouse, with a strict leakage audit.

In [9]:
# Label-trap guard: trend_direction / trend_pct derive the label, so never features.
label_columns = {"trend_direction", "trend_pct"}
feature_candidates = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr",
                      "avg_position", "engagement_rate", "content_age_days",
                      "word_count", "days_with_impressions"]
leak = [c for c in feature_candidates if c in label_columns]
print("Candidate feature columns:", feature_candidates)
print("Label-derived columns sneaking into features:", leak if leak else "none")


Candidate feature columns: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'word_count', 'days_with_impressions']
Label-derived columns sneaking into features: none


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.